# 🧬 Dark Manifold V21 — Biological Foundation Model

## A Paradigm Shift: From Pattern Matching to Biological Reasoning

V21 isn't just "better" than V20 — it's a fundamentally different approach.

### The 10 New Mechanisms

| # | Mechanism | What It Does | Biological Equivalent |
|---|-----------|--------------|----------------------|
| 1 | **Transformer Encoder** | Self-attention over ALL inputs | Global cellular awareness |
| 2 | **Neural ODE** | Continuous-time dynamics | Real kinetics, not discrete |
| 3 | **Enzyme Kinetics Layer** | Michaelis-Menten decoder | Actual Vmax, Km parameters |
| 4 | **Hyperbolic Embeddings** | Hierarchy in Poincaré ball | Pathway organization |
| 5 | **Gene Regulatory Network** | TF binding dynamics | Transcriptional control |
| 6 | **Causal Discovery** | Learn intervention effects | "What if we knockout X?" |
| 7 | **Meta-Learning (MAML)** | Few-shot adaptation | Bacterial plasticity |
| 8 | **World Model** | Latent dreaming | Cellular anticipation |
| 9 | **Active Inference** | Free energy goals | Homeostatic drive |
| 10 | **Compositional Modules** | Plug-and-play pathways | Modular metabolism |

---

## Key Insight

V16-V20 tried to **learn the mapping** from conditions to metabolites.

V21 tries to **understand the system** — it builds a causal, hierarchical, 
goal-directed model of how cells actually work.

In [ ]:
#@title 1️⃣ Setup & Imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import grad
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import copy
import time
import json
import math
import warnings
warnings.filterwarnings('ignore')

# Install torchdiffeq for Neural ODE
try:
    from torchdiffeq import odeint, odeint_adjoint
except ImportError:
    !pip install torchdiffeq -q
    from torchdiffeq import odeint, odeint_adjoint

# Install torch_geometric for GNN
try:
    from torch_geometric.nn import GATConv
except ImportError:
    !pip install torch_geometric -q
    from torch_geometric.nn import GATConv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
print(f"🔥 PyTorch: {torch.__version__}")

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
#@title 2️⃣ Configuration

# Core dimensions
N_METABOLITES = 50
N_GENES = 30  # NEW: Gene expression layer
PERTURBATION_DIM = 10  # Extended

# Transformer
D_MODEL = 128
N_HEADS = 8
N_TRANSFORMER_LAYERS = 6
D_FF = 256
DROPOUT = 0.1

# Neural ODE
LATENT_DIM = 64
ODE_HIDDEN = 128

# Hyperbolic
HYPERBOLIC_DIM = 32
CURVATURE = 1.0

# World Model
WORLD_DIM = 64

# Training
N_EPOCHS = 2000
LEARNING_RATE = 3e-4
BATCH_SIZE = 4
META_LR = 1e-3  # MAML outer loop
INNER_LR = 0.01  # MAML inner loop
N_INNER_STEPS = 5  # MAML adaptation steps

# Loss weights
LAMBDA_RECON = 1.0
LAMBDA_KL = 0.1  # World model KL
LAMBDA_CAUSAL = 0.2
LAMBDA_FREE_ENERGY = 0.1

print("📋 V21 Configuration:")
print(f"   Transformer: {N_TRANSFORMER_LAYERS} layers, {N_HEADS} heads")
print(f"   Neural ODE latent dim: {LATENT_DIM}")
print(f"   Hyperbolic dim: {HYPERBOLIC_DIM}")
print(f"   Gene regulatory: {N_GENES} genes")

In [ ]:
#@title 3️⃣ Mechanism 4: Hyperbolic Embeddings (Poincaré Ball)

class HyperbolicLayer(nn.Module):
    """
    Embeddings in hyperbolic space (Poincaré ball model).
    
    Why hyperbolic? Metabolic networks have HIERARCHY:
    - Glucose → G6P → Glycolysis intermediates → Pyruvate → TCA
    - This is a tree-like structure
    - Hyperbolic space naturally encodes trees with bounded distortion
    
    In Euclidean space, you need exponentially more dimensions to embed trees.
    In hyperbolic space, you can embed infinitely deep trees in 2D.
    """
    
    def __init__(self, in_dim, hyp_dim, c=1.0):
        super().__init__()
        self.c = c  # Curvature
        self.hyp_dim = hyp_dim
        
        # Euclidean -> Hyperbolic projection
        self.proj = nn.Linear(in_dim, hyp_dim)
    
    def exp_map(self, v, x=None):
        """
        Exponential map: Tangent space -> Poincaré ball.
        Maps a Euclidean vector to a point on the manifold.
        """
        if x is None:
            x = torch.zeros_like(v)
        
        c = self.c
        v_norm = torch.clamp(v.norm(dim=-1, keepdim=True), min=1e-7)
        
        # Möbius addition formula
        lambda_x = 2 / (1 - c * (x * x).sum(dim=-1, keepdim=True).clamp(max=1-1e-5))
        
        # Exponential map at origin
        sqrt_c = math.sqrt(c)
        result = torch.tanh(sqrt_c * lambda_x * v_norm / 2) * v / (sqrt_c * v_norm)
        
        # Clamp to ball interior
        norm = result.norm(dim=-1, keepdim=True)
        max_norm = (1 - 1e-5) / math.sqrt(c)
        result = result / norm.clamp(min=1e-7) * norm.clamp(max=max_norm)
        
        return result
    
    def hyperbolic_distance(self, x, y):
        """
        Distance in Poincaré ball.
        Key property: distance grows exponentially as you move toward the boundary.
        This is what makes it perfect for hierarchies.
        """
        c = self.c
        sqrt_c = math.sqrt(c)
        
        diff = x - y
        diff_norm_sq = (diff * diff).sum(dim=-1).clamp(min=1e-10)
        x_norm_sq = (x * x).sum(dim=-1).clamp(max=1-1e-5)
        y_norm_sq = (y * y).sum(dim=-1).clamp(max=1-1e-5)
        
        numerator = 2 * diff_norm_sq
        denominator = (1 - c * x_norm_sq) * (1 - c * y_norm_sq)
        
        arg = 1 + numerator / denominator.clamp(min=1e-10)
        dist = (1 / sqrt_c) * torch.acosh(arg.clamp(min=1.0 + 1e-7))
        
        return dist
    
    def forward(self, x):
        """Project to hyperbolic space."""
        # Euclidean projection
        v = self.proj(x)
        
        # Map to Poincaré ball
        h = self.exp_map(v)
        
        return h

print("✅ HyperbolicLayer (Poincaré ball) defined")

In [ ]:
#@title 4️⃣ Mechanism 3: Enzyme Kinetics Layer

class EnzymeKineticsLayer(nn.Module):
    """
    Michaelis-Menten kinetics built directly into the network.
    
    Instead of learning arbitrary functions, we constrain the network
    to follow actual enzyme behavior:
    
    v = Vmax * S / (Km + S)  [basic]
    v = Vmax * S / (Km * (1 + I/Ki) + S)  [competitive inhibition]
    v = Vmax * S / (Km + S * (1 + I/Ki))  [uncompetitive inhibition]
    
    The network learns Vmax, Km, Ki parameters rather than arbitrary weights.
    This makes predictions biologically interpretable.
    """
    
    def __init__(self, n_enzymes, n_metabolites, hidden_dim):
        super().__init__()
        self.n_enzymes = n_enzymes
        self.n_metabolites = n_metabolites
        
        # Learnable kinetic parameters
        # Vmax: maximum velocity (positive)
        self.log_Vmax = nn.Parameter(torch.zeros(n_enzymes))
        
        # Km: Michaelis constant (positive)
        self.log_Km = nn.Parameter(torch.zeros(n_enzymes))
        
        # Substrate specificity: which metabolite is substrate for which enzyme
        self.substrate_weights = nn.Parameter(torch.randn(n_enzymes, n_metabolites) * 0.1)
        
        # Inhibition: which metabolites inhibit which enzymes
        self.log_Ki = nn.Parameter(torch.ones(n_enzymes, n_metabolites))  # Inhibition constants
        
        # Environment modulation of kinetics
        self.env_modulator = nn.Sequential(
            nn.Linear(hidden_dim, n_enzymes * 2),  # Modulate Vmax and Km
            nn.Tanh()
        )
    
    def forward(self, metabolites, env_state):
        """
        Compute reaction velocities using Michaelis-Menten.
        
        Args:
            metabolites: (batch, n_metabolites) - substrate concentrations
            env_state: (batch, hidden_dim) - environment encoding
        
        Returns:
            velocities: (batch, n_enzymes) - reaction rates
        """
        batch_size = metabolites.shape[0]
        
        # Base kinetic parameters
        Vmax = torch.exp(self.log_Vmax).unsqueeze(0)  # (1, n_enzymes)
        Km = torch.exp(self.log_Km).unsqueeze(0)  # (1, n_enzymes)
        Ki = torch.exp(self.log_Ki)  # (n_enzymes, n_metabolites)
        
        # Environment modulation
        env_mod = self.env_modulator(env_state)  # (batch, n_enzymes * 2)
        vmax_mod = env_mod[:, :self.n_enzymes]  # (batch, n_enzymes)
        km_mod = env_mod[:, self.n_enzymes:]  # (batch, n_enzymes)
        
        # Modulated parameters
        Vmax_eff = Vmax * (1 + 0.5 * vmax_mod)  # Scale Vmax by environment
        Km_eff = Km * torch.exp(0.5 * km_mod)  # Scale Km by environment
        
        # Compute effective substrate concentration for each enzyme
        # Weighted sum of metabolites (sparse in reality)
        substrate_probs = F.softmax(self.substrate_weights, dim=1)  # (n_enzymes, n_metabolites)
        S = torch.matmul(metabolites, substrate_probs.T)  # (batch, n_enzymes)
        
        # Compute inhibition term
        # Sum of [metabolite]/Ki for each inhibitor
        inhibition = torch.matmul(metabolites, (1.0 / Ki).T)  # (batch, n_enzymes)
        
        # Michaelis-Menten with competitive inhibition
        # v = Vmax * S / (Km * (1 + sum(I/Ki)) + S)
        v = Vmax_eff * S / (Km_eff * (1 + inhibition) + S + 1e-8)
        
        return v

print("✅ EnzymeKineticsLayer (Michaelis-Menten) defined")

In [ ]:
#@title 5️⃣ Mechanism 5: Gene Regulatory Network

class GeneRegulatoryNetwork(nn.Module):
    """
    Models gene expression dynamics.
    
    Central dogma: DNA -> mRNA -> Protein -> Metabolite
    
    Key components:
    1. Transcription factors (TFs) bind to promoters
    2. TF binding activates or represses transcription
    3. mRNA is translated to protein
    4. Proteins (enzymes) catalyze metabolic reactions
    
    This is much SLOWER than metabolic dynamics (minutes-hours vs seconds).
    """
    
    def __init__(self, n_genes, n_metabolites, hidden_dim):
        super().__init__()
        self.n_genes = n_genes
        self.n_metabolites = n_metabolites
        
        # TF-gene interaction matrix (sparse in reality)
        # Positive = activation, negative = repression
        self.tf_gene_matrix = nn.Parameter(torch.randn(n_genes, n_genes) * 0.1)
        
        # Metabolite -> TF activity (some metabolites activate TFs)
        # e.g., cAMP activates CRP, ppGpp activates stringent response
        self.metabolite_to_tf = nn.Linear(n_metabolites, n_genes)
        
        # Environment -> gene expression (stress response)
        self.env_to_gene = nn.Linear(hidden_dim, n_genes)
        
        # Gene -> enzyme/metabolite effect
        self.gene_to_metabolite = nn.Linear(n_genes, n_metabolites)
        
        # Transcription/translation timescales
        self.log_tau_transcription = nn.Parameter(torch.log(torch.tensor(0.5)))  # ~30 min
        self.log_tau_translation = nn.Parameter(torch.log(torch.tensor(0.2)))  # ~10 min
        
        # mRNA decay
        self.log_mrna_decay = nn.Parameter(torch.log(torch.tensor(0.1)))  # ~5 min half-life
    
    def compute_tf_activity(self, gene_expression, metabolites, env_state):
        """
        Compute transcription factor activities.
        
        TF activity depends on:
        1. Gene expression of the TF itself
        2. Metabolite signals (e.g., cAMP)
        3. Environmental stress signals
        """
        # Base TF activity from gene expression
        tf_base = gene_expression
        
        # Metabolite modulation
        tf_metabolite = torch.tanh(self.metabolite_to_tf(metabolites))
        
        # Environment modulation
        tf_env = torch.tanh(self.env_to_gene(env_state))
        
        # Combined TF activity (multiplicative gating)
        tf_activity = tf_base * (1 + tf_metabolite) * (1 + tf_env)
        
        return tf_activity
    
    def forward(self, gene_expression, metabolites, env_state, dt):
        """
        Update gene expression.
        
        dG/dt = (1/tau_trans) * f(TF_activity) - decay * G
        
        Returns:
            new_gene_expr: Updated gene expression levels
            metabolite_effect: Effect on metabolite dynamics
        """
        # Compute TF activities
        tf_activity = self.compute_tf_activity(gene_expression, metabolites, env_state)
        
        # TF -> target gene regulation
        # Each gene's expression is influenced by TFs
        regulation = torch.tanh(torch.matmul(tf_activity, self.tf_gene_matrix.T))
        
        # Gene expression dynamics
        tau = torch.exp(self.log_tau_transcription)
        decay = torch.exp(self.log_mrna_decay)
        
        # Hill-function style activation
        activation = torch.sigmoid(regulation)  # 0-1 activation
        
        # dG/dt = production - decay
        dG_dt = (activation - gene_expression) / tau - decay * gene_expression
        
        # Update
        new_gene_expr = gene_expression + dt * dG_dt
        new_gene_expr = torch.clamp(new_gene_expr, 0.01, 10.0)
        
        # Gene expression affects metabolites (via enzymes)
        metabolite_effect = self.gene_to_metabolite(new_gene_expr)
        
        return new_gene_expr, metabolite_effect

print("✅ GeneRegulatoryNetwork defined")

In [ ]:
#@title 6️⃣ Mechanism 2: Neural ODE Dynamics

class ODEFunc(nn.Module):
    """
    The dynamics function for Neural ODE.
    
    Instead of discrete time steps: x_{t+1} = f(x_t)
    We model continuous dynamics: dx/dt = f(x, t)
    
    This is solved with adaptive ODE solvers (Dormand-Prince, etc.)
    which automatically adjust step size for accuracy.
    """
    
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),  # +1 for time
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, latent_dim)
        )
    
    def forward(self, t, z):
        """
        Compute dz/dt.
        
        Args:
            t: Current time (scalar)
            z: Current state (batch, latent_dim)
        """
        # Concatenate time to state
        t_vec = t.expand(z.shape[0], 1) if z.dim() > 1 else t.unsqueeze(0)
        z_t = torch.cat([z, t_vec], dim=-1)
        
        return self.net(z_t)


class NeuralODE(nn.Module):
    """
    Full Neural ODE module.
    
    Benefits:
    1. Constant memory regardless of integration time (adjoint method)
    2. Adaptive computation - more steps where dynamics are fast
    3. Continuous-time predictions at any resolution
    4. Better generalization to unseen time horizons
    """
    
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.func = ODEFunc(latent_dim, hidden_dim)
    
    def forward(self, z0, t_span):
        """
        Integrate the ODE from t_span[0] to t_span[-1].
        
        Args:
            z0: Initial state (batch, latent_dim)
            t_span: Times to evaluate (n_times,)
        
        Returns:
            z_t: States at each time (n_times, batch, latent_dim)
        """
        # Use adjoint method for memory efficiency
        z_t = odeint_adjoint(
            self.func, z0, t_span,
            method='dopri5',  # Adaptive Runge-Kutta
            rtol=1e-4, atol=1e-5
        )
        
        return z_t  # (n_times, batch, latent_dim)

print("✅ NeuralODE defined")

In [ ]:
#@title 7️⃣ Mechanism 8: World Model (Latent Dynamics)

class WorldModel(nn.Module):
    """
    Learns a latent space model of cellular dynamics.
    
    Inspired by:
    - World Models (Ha & Schmidhuber)
    - Dreamer (Hafner et al.)
    
    The cell can "dream" - predict future states without
    actually running the full simulation.
    
    Components:
    1. Encoder: x -> z (compress to latent)
    2. Prior: p(z_t | z_{t-1}) (predict next latent)
    3. Posterior: q(z_t | z_{t-1}, x_t) (infer latent from obs)
    4. Decoder: z -> x (reconstruct observation)
    """
    
    def __init__(self, obs_dim, latent_dim, hidden_dim):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder: observation -> latent
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim * 2)  # Mean and log-var
        )
        
        # Prior: predict next latent from current
        self.prior = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim * 2)  # Mean and log-var
        )
        
        # Posterior: infer latent from previous latent + observation
        self.posterior = nn.Sequential(
            nn.Linear(latent_dim + obs_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim * 2)
        )
        
        # Decoder: latent -> observation
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, obs_dim)
        )
    
    def reparameterize(self, mean, logvar):
        """Reparameterization trick for VAE."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std
    
    def encode(self, x):
        """Encode observation to latent."""
        h = self.encoder(x)
        mean, logvar = h.chunk(2, dim=-1)
        logvar = torch.clamp(logvar, -10, 2)
        z = self.reparameterize(mean, logvar)
        return z, mean, logvar
    
    def predict_prior(self, z_prev):
        """Predict next latent (dreaming)."""
        h = self.prior(z_prev)
        mean, logvar = h.chunk(2, dim=-1)
        logvar = torch.clamp(logvar, -10, 2)
        return mean, logvar
    
    def infer_posterior(self, z_prev, x):
        """Infer latent from observation (grounded)."""
        h = self.posterior(torch.cat([z_prev, x], dim=-1))
        mean, logvar = h.chunk(2, dim=-1)
        logvar = torch.clamp(logvar, -10, 2)
        return mean, logvar
    
    def decode(self, z):
        """Decode latent to observation."""
        return self.decoder(z)
    
    def dream(self, z0, n_steps):
        """
        Dream forward in latent space.
        Predict future without observations.
        """
        z = z0
        z_sequence = [z]
        
        for _ in range(n_steps):
            prior_mean, prior_logvar = self.predict_prior(z)
            z = self.reparameterize(prior_mean, prior_logvar)
            z_sequence.append(z)
        
        return torch.stack(z_sequence, dim=1)  # (batch, n_steps+1, latent_dim)
    
    def forward(self, x_sequence):
        """
        Forward pass through sequence.
        
        Returns reconstructions, KL divergences, and latents.
        """
        batch_size, seq_len, _ = x_sequence.shape
        
        # Initialize
        z, post_mean, post_logvar = self.encode(x_sequence[:, 0])
        
        recons = [self.decode(z)]
        kl_divs = []
        latents = [z]
        
        for t in range(1, seq_len):
            # Prior: predict from previous latent
            prior_mean, prior_logvar = self.predict_prior(z)
            
            # Posterior: infer from observation
            post_mean, post_logvar = self.infer_posterior(z, x_sequence[:, t])
            
            # KL divergence between posterior and prior
            kl = 0.5 * (prior_logvar - post_logvar + 
                       (torch.exp(post_logvar) + (post_mean - prior_mean)**2) / 
                       torch.exp(prior_logvar) - 1)
            kl_divs.append(kl.sum(dim=-1))
            
            # Sample from posterior
            z = self.reparameterize(post_mean, post_logvar)
            latents.append(z)
            
            # Decode
            recons.append(self.decode(z))
        
        recons = torch.stack(recons, dim=1)  # (batch, seq_len, obs_dim)
        kl_divs = torch.stack(kl_divs, dim=1) if kl_divs else torch.zeros(batch_size, 1)
        latents = torch.stack(latents, dim=1)
        
        return recons, kl_divs, latents

print("✅ WorldModel (latent dynamics) defined")

In [ ]:
#@title 8️⃣ Mechanism 9: Active Inference Controller

class ActiveInference(nn.Module):
    """
    Active Inference / Free Energy Principle.
    
    The cell has GOALS (homeostasis) and takes actions to minimize
    "surprise" (free energy).
    
    Free energy F ≈ prediction error + complexity
    F = E_q[log q(z) - log p(z,x)] 
      = KL[q(z)||p(z)] - E_q[log p(x|z)]
    
    The cell "acts" by adjusting its metabolism to minimize F.
    This is a mathematical formalization of homeostasis.
    """
    
    def __init__(self, latent_dim, action_dim, hidden_dim):
        super().__init__()
        
        self.latent_dim = latent_dim
        self.action_dim = action_dim
        
        # Preferences: what states does the cell "want"?
        # These are homeostatic setpoints
        self.preferred_state = nn.Parameter(torch.zeros(latent_dim))
        self.preference_precision = nn.Parameter(torch.ones(latent_dim))  # How strongly?
        
        # Policy: state -> action (metabolic adjustments)
        self.policy = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Tanh()  # Bounded actions
        )
        
        # Expected free energy predictor
        self.efe_predictor = nn.Sequential(
            nn.Linear(latent_dim + action_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def compute_free_energy(self, z, z_pred, obs, obs_pred):
        """
        Compute variational free energy.
        
        F = prediction_error + complexity
        """
        # Prediction error (observation vs prediction)
        obs_error = ((obs - obs_pred) ** 2).sum(dim=-1)
        
        # State prediction error
        state_error = ((z - z_pred) ** 2).sum(dim=-1)
        
        # Preference error (how far from preferred state?)
        precision = F.softplus(self.preference_precision)
        pref_error = (precision * (z - self.preferred_state) ** 2).sum(dim=-1)
        
        # Total free energy
        F = obs_error + 0.1 * state_error + 0.5 * pref_error
        
        return F
    
    def select_action(self, z):
        """
        Select action to minimize expected free energy.
        """
        # Direct policy
        action = self.policy(z)
        
        # Compute expected free energy for this action
        efe_input = torch.cat([z, action], dim=-1)
        expected_fe = self.efe_predictor(efe_input)
        
        return action, expected_fe
    
    def forward(self, z, z_pred=None, obs=None, obs_pred=None):
        """
        Compute action and free energy.
        """
        action, expected_fe = self.select_action(z)
        
        if z_pred is not None and obs is not None and obs_pred is not None:
            actual_fe = self.compute_free_energy(z, z_pred, obs, obs_pred)
        else:
            actual_fe = expected_fe.squeeze(-1)
        
        return action, actual_fe, expected_fe.squeeze(-1)

print("✅ ActiveInference (free energy) defined")

In [ ]:
#@title 9️⃣ Mechanism 6: Causal Discovery Layer

class CausalLayer(nn.Module):
    """
    Learn causal structure and intervention effects.
    
    Key insight: Correlation ≠ Causation
    
    The cell needs to understand:
    - "If I knock out gene X, what happens?" (intervention)
    - "What would have happened if X were different?" (counterfactual)
    
    We use structural causal models:
    - Each variable X_i = f_i(Pa(X_i), noise_i)
    - Interventions do(X=x) break the natural mechanism
    """
    
    def __init__(self, n_vars, hidden_dim):
        super().__init__()
        self.n_vars = n_vars
        
        # Learnable causal graph (adjacency matrix)
        # A[i,j] = effect of variable j on variable i
        self.causal_weights = nn.Parameter(torch.randn(n_vars, n_vars) * 0.1)
        
        # Mask to enforce DAG (no cycles) - upper triangular
        self.register_buffer('dag_mask', torch.triu(torch.ones(n_vars, n_vars), diagonal=1))
        
        # Structural equations: f_i(Pa(X_i))
        self.structural_eqs = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_vars, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, 1)
            )
            for _ in range(n_vars)
        ])
    
    def get_causal_graph(self):
        """Get the learned causal adjacency matrix."""
        # Apply DAG constraint
        A = torch.sigmoid(self.causal_weights) * self.dag_mask
        return A
    
    def intervene(self, x, intervention_idx, intervention_value):
        """
        Perform intervention do(X_i = v).
        
        This "breaks" the natural mechanism for X_i
        and sets it to a fixed value.
        """
        x_intervened = x.clone()
        x_intervened[:, intervention_idx] = intervention_value
        return x_intervened
    
    def forward(self, x, intervention=None):
        """
        Forward pass with optional intervention.
        
        Args:
            x: Input variables (batch, n_vars)
            intervention: Dict of {var_idx: value} or None
        
        Returns:
            y: Output after causal propagation
            A: Causal adjacency matrix
        """
        A = self.get_causal_graph()
        
        # Apply intervention if specified
        if intervention is not None:
            for var_idx, value in intervention.items():
                x = self.intervene(x, var_idx, value)
                # Zero out incoming edges to intervened variable
                A = A.clone()
                A[var_idx, :] = 0
        
        # Propagate through causal graph
        # Weight input by causal structure
        weighted_input = x @ A.T  # (batch, n_vars)
        
        # Apply structural equations
        outputs = []
        for i, f_i in enumerate(self.structural_eqs):
            # Parents contribute to this variable
            parent_input = weighted_input * A[i, :].unsqueeze(0)
            y_i = f_i(parent_input + x)  # Add direct effect
            outputs.append(y_i)
        
        y = torch.cat(outputs, dim=-1)
        
        return y, A
    
    def counterfactual(self, x, intervention):
        """
        Compute counterfactual: "What would have happened if...?"
        
        Steps:
        1. Abduction: Infer noise terms from observed data
        2. Action: Apply intervention
        3. Prediction: Propagate with fixed noise
        """
        # Get prediction under intervention
        y_cf, _ = self.forward(x, intervention)
        return y_cf

print("✅ CausalLayer (do-calculus) defined")

In [ ]:
#@title 🔟 Mechanism 1: Transformer Encoder

class BiologicalTransformer(nn.Module):
    """
    Transformer encoder for biological state.
    
    Unlike GNN which is local (only connected metabolites talk),
    Transformer allows GLOBAL attention:
    - Every metabolite can attend to every other
    - Environment can attend to all metabolites at once
    - Genes can attend to relevant metabolites
    
    This captures long-range dependencies in metabolism.
    """
    
    def __init__(self, n_metabolites, n_genes, perturbation_dim, 
                 d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        
        self.d_model = d_model
        
        # Embeddings for different input types
        self.metabolite_embed = nn.Linear(1, d_model)
        self.gene_embed = nn.Linear(1, d_model)
        self.perturbation_embed = nn.Linear(perturbation_dim, d_model)
        
        # Type embeddings (metabolite vs gene vs environment)
        self.type_embed = nn.Embedding(3, d_model)
        
        # Position embeddings for metabolites/genes
        self.pos_embed_met = nn.Embedding(n_metabolites, d_model)
        self.pos_embed_gene = nn.Embedding(n_genes, d_model)
        
        # Transformer layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-LN for stability
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, metabolites, genes, perturbation):
        """
        Args:
            metabolites: (batch, n_metabolites)
            genes: (batch, n_genes)
            perturbation: (batch, perturbation_dim)
        
        Returns:
            encoded: (batch, n_tokens, d_model)
        """
        batch_size = metabolites.shape[0]
        n_met = metabolites.shape[1]
        n_gene = genes.shape[1]
        device = metabolites.device
        
        # Embed metabolites
        met_tokens = self.metabolite_embed(metabolites.unsqueeze(-1))  # (batch, n_met, d_model)
        met_pos = self.pos_embed_met(torch.arange(n_met, device=device))  # (n_met, d_model)
        met_type = self.type_embed(torch.zeros(n_met, dtype=torch.long, device=device))  # (n_met, d_model)
        met_tokens = met_tokens + met_pos.unsqueeze(0) + met_type.unsqueeze(0)
        
        # Embed genes
        gene_tokens = self.gene_embed(genes.unsqueeze(-1))  # (batch, n_gene, d_model)
        gene_pos = self.pos_embed_gene(torch.arange(n_gene, device=device))
        gene_type = self.type_embed(torch.ones(n_gene, dtype=torch.long, device=device))
        gene_tokens = gene_tokens + gene_pos.unsqueeze(0) + gene_type.unsqueeze(0)
        
        # Embed perturbation (single token)
        pert_token = self.perturbation_embed(perturbation).unsqueeze(1)  # (batch, 1, d_model)
        pert_type = self.type_embed(torch.tensor([2], device=device))  # (1, d_model)
        pert_token = pert_token + pert_type.unsqueeze(0)
        
        # Concatenate all tokens
        tokens = torch.cat([pert_token, met_tokens, gene_tokens], dim=1)  # (batch, 1+n_met+n_gene, d_model)
        
        # Transformer encoding
        encoded = self.transformer(tokens)
        encoded = self.norm(encoded)
        
        return encoded

print("✅ BiologicalTransformer defined")

In [ ]:
#@title 1️⃣1️⃣ Mechanism 7: Meta-Learning (MAML)

class MAMLWrapper:
    """
    Model-Agnostic Meta-Learning (MAML) for few-shot adaptation.
    
    The idea: Learn initial parameters that can quickly adapt
    to new conditions with just a few gradient steps.
    
    This mimics bacterial adaptation:
    - Bacteria have evolved to quickly respond to new environments
    - They don't start from scratch - they have "priors" for adaptation
    
    MAML learns these priors.
    """
    
    def __init__(self, model, inner_lr=0.01, n_inner_steps=5):
        self.model = model
        self.inner_lr = inner_lr
        self.n_inner_steps = n_inner_steps
    
    def inner_loop(self, support_x, support_y, create_graph=True):
        """
        Adapt to a new task using support set.
        
        Args:
            support_x: Support examples (few examples of new condition)
            support_y: Support labels
            create_graph: Whether to track gradients through adaptation
        
        Returns:
            Adapted model parameters
        """
        # Clone model for adaptation
        adapted_params = {name: param.clone() for name, param in self.model.named_parameters()}
        
        # Inner loop: gradient descent on support set
        for step in range(self.n_inner_steps):
            # Forward pass with current adapted params
            pred = self._forward_with_params(support_x, adapted_params)
            loss = F.mse_loss(pred, support_y)
            
            # Compute gradients
            grads = torch.autograd.grad(
                loss, adapted_params.values(),
                create_graph=create_graph,
                allow_unused=True
            )
            
            # Update adapted parameters
            adapted_params = {
                name: param - self.inner_lr * (grad if grad is not None else torch.zeros_like(param))
                for (name, param), grad in zip(adapted_params.items(), grads)
            }
        
        return adapted_params
    
    def _forward_with_params(self, x, params):
        """
        Forward pass using specific parameters.
        This is needed because we use cloned parameters.
        """
        # This is a simplified version - in practice you'd use
        # functional form of the model
        # For now, we'll use a workaround
        
        # Save original params
        original_params = {name: param.clone() for name, param in self.model.named_parameters()}
        
        # Load adapted params
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                param.copy_(params[name])
        
        # Forward
        output = self.model(x)
        
        # Restore original
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                param.copy_(original_params[name])
        
        return output
    
    def outer_loss(self, tasks):
        """
        Compute meta-loss across tasks.
        
        Args:
            tasks: List of (support_x, support_y, query_x, query_y) tuples
        
        Returns:
            Meta-loss to backpropagate
        """
        meta_loss = 0.0
        
        for support_x, support_y, query_x, query_y in tasks:
            # Adapt to task
            adapted_params = self.inner_loop(support_x, support_y)
            
            # Evaluate on query set
            pred = self._forward_with_params(query_x, adapted_params)
            meta_loss = meta_loss + F.mse_loss(pred, query_y)
        
        return meta_loss / len(tasks)

print("✅ MAML (Meta-Learning) wrapper defined")

In [ ]:
#@title 1️⃣2️⃣ Mechanism 10: Compositional Modules

class PathwayModule(nn.Module):
    """
    A single metabolic pathway as a composable module.
    
    Pathways are like LEGO blocks:
    - Glycolysis: Glucose -> Pyruvate
    - TCA: Acetyl-CoA -> CO2 + energy
    - PPP: Glucose-6P -> NADPH + Ribose-5P
    
    Each module has:
    - Inputs (substrates)
    - Outputs (products)
    - Internal state
    - Interface to connect with other modules
    """
    
    def __init__(self, name, n_internal, n_inputs, n_outputs, hidden_dim):
        super().__init__()
        self.name = name
        self.n_internal = n_internal
        
        # Internal dynamics
        self.internal_net = nn.Sequential(
            nn.Linear(n_internal + n_inputs, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_internal)
        )
        
        # Output production
        self.output_net = nn.Sequential(
            nn.Linear(n_internal, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_outputs)
        )
        
        # Flux control (regulation)
        self.flux_gate = nn.Sequential(
            nn.Linear(n_inputs, n_internal),
            nn.Sigmoid()
        )
    
    def forward(self, internal_state, inputs):
        """
        One step of pathway dynamics.
        
        Args:
            internal_state: (batch, n_internal) - pathway intermediates
            inputs: (batch, n_inputs) - substrates from other pathways
        
        Returns:
            new_internal: Updated internal state
            outputs: Products to send to other pathways
        """
        # Flux control based on inputs
        gate = self.flux_gate(inputs)
        
        # Internal dynamics
        combined = torch.cat([internal_state, inputs], dim=-1)
        delta = self.internal_net(combined)
        new_internal = internal_state + gate * delta
        new_internal = torch.clamp(new_internal, 0.01, 10.0)
        
        # Produce outputs
        outputs = F.relu(self.output_net(new_internal))
        
        return new_internal, outputs


class CompositionalMetabolism(nn.Module):
    """
    Full metabolism as composition of pathway modules.
    
    This is more biologically realistic because:
    1. Pathways are semi-autonomous (like in real cells)
    2. They communicate through shared metabolites
    3. New pathways can be added without retraining everything
    4. Transfer learning: glycolysis module can transfer to any organism
    """
    
    def __init__(self, hidden_dim):
        super().__init__()
        
        # Define pathway modules
        self.glycolysis = PathwayModule('glycolysis', n_internal=8, n_inputs=3, n_outputs=3, hidden_dim=hidden_dim)
        self.tca = PathwayModule('tca', n_internal=7, n_inputs=3, n_outputs=4, hidden_dim=hidden_dim)
        self.ppp = PathwayModule('ppp', n_internal=4, n_inputs=2, n_outputs=3, hidden_dim=hidden_dim)
        self.aa_synthesis = PathwayModule('aa_synthesis', n_internal=10, n_inputs=5, n_outputs=5, hidden_dim=hidden_dim)
        self.energy = PathwayModule('energy', n_internal=4, n_inputs=4, n_outputs=2, hidden_dim=hidden_dim)
        
        # Connection matrix: which outputs feed into which inputs
        # This defines the metabolic network topology
        self.connection_weights = nn.ParameterDict({
            'glyc_to_tca': nn.Parameter(torch.randn(3, 3) * 0.1),
            'glyc_to_ppp': nn.Parameter(torch.randn(3, 2) * 0.1),
            'tca_to_aa': nn.Parameter(torch.randn(4, 5) * 0.1),
            'ppp_to_aa': nn.Parameter(torch.randn(3, 5) * 0.1),
            'tca_to_energy': nn.Parameter(torch.randn(4, 4) * 0.1),
        })
    
    def forward(self, external_glucose, external_signals, states):
        """
        One step of composed metabolism.
        
        Args:
            external_glucose: (batch, 1) - glucose input
            external_signals: (batch, 5) - environmental signals
            states: Dict of internal states for each pathway
        
        Returns:
            new_states: Updated pathway states
            outputs: Dict of pathway outputs
        """
        # Glycolysis: glucose -> pyruvate, ATP, NADH
        glyc_input = torch.cat([external_glucose, external_signals[:, :2]], dim=-1)
        states['glycolysis'], glyc_out = self.glycolysis(states['glycolysis'], glyc_input)
        
        # PPP: G6P -> NADPH, R5P
        ppp_input = glyc_out[:, :2] @ self.connection_weights['glyc_to_ppp'].T
        states['ppp'], ppp_out = self.ppp(states['ppp'], ppp_input)
        
        # TCA: pyruvate -> CO2, NADH, FADH2
        tca_input = glyc_out @ self.connection_weights['glyc_to_tca'].T
        states['tca'], tca_out = self.tca(states['tca'], tca_input)
        
        # AA synthesis: TCA intermediates + PPP -> amino acids
        aa_input = (tca_out @ self.connection_weights['tca_to_aa'].T + 
                   ppp_out @ self.connection_weights['ppp_to_aa'].T)
        states['aa_synthesis'], aa_out = self.aa_synthesis(states['aa_synthesis'], aa_input)
        
        # Energy: NADH, FADH2 -> ATP
        energy_input = tca_out @ self.connection_weights['tca_to_energy'].T
        states['energy'], energy_out = self.energy(states['energy'], energy_input)
        
        outputs = {
            'glycolysis': glyc_out,
            'ppp': ppp_out,
            'tca': tca_out,
            'aa': aa_out,
            'energy': energy_out
        }
        
        return states, outputs
    
    def init_states(self, batch_size, device):
        """Initialize all pathway states."""
        return {
            'glycolysis': torch.ones(batch_size, 8, device=device),
            'tca': torch.ones(batch_size, 7, device=device),
            'ppp': torch.ones(batch_size, 4, device=device),
            'aa_synthesis': torch.ones(batch_size, 10, device=device),
            'energy': torch.ones(batch_size, 4, device=device)
        }

print("✅ CompositionalMetabolism (modular pathways) defined")

In [ ]:
#@title 1️⃣3️⃣ Complete V21 Model

class DarkManifoldV21(nn.Module):
    """
    V21: Biological Foundation Model
    
    Combines all 10 mechanisms:
    1. Transformer encoder (global attention)
    2. Neural ODE (continuous dynamics)
    3. Enzyme kinetics (Michaelis-Menten)
    4. Hyperbolic embeddings (hierarchy)
    5. Gene regulatory network (transcription)
    6. Causal discovery (interventions)
    7. Meta-learning (MAML, handled externally)
    8. World model (latent dreaming)
    9. Active inference (goal-directed)
    10. Compositional modules (pathways)
    """
    
    def __init__(self, config):
        super().__init__()
        
        self.config = config
        n_met = config['n_metabolites']
        n_gene = config['n_genes']
        p_dim = config['perturbation_dim']
        d_model = config['d_model']
        latent_dim = config['latent_dim']
        hidden = config['hidden_dim']
        
        # 1. Transformer encoder
        self.transformer = BiologicalTransformer(
            n_metabolites=n_met,
            n_genes=n_gene,
            perturbation_dim=p_dim,
            d_model=d_model,
            n_heads=config['n_heads'],
            n_layers=config['n_transformer_layers'],
            d_ff=config['d_ff'],
            dropout=config['dropout']
        )
        
        # 2. Neural ODE
        self.neural_ode = NeuralODE(latent_dim, hidden)
        
        # 3. Enzyme kinetics
        self.enzyme_kinetics = EnzymeKineticsLayer(n_met, n_met, d_model)
        
        # 4. Hyperbolic embeddings
        self.hyperbolic = HyperbolicLayer(d_model, config['hyperbolic_dim'])
        
        # 5. Gene regulatory network
        self.grn = GeneRegulatoryNetwork(n_gene, n_met, d_model)
        
        # 6. Causal layer
        self.causal = CausalLayer(n_met, hidden)
        
        # 8. World model
        obs_dim = n_met + n_gene
        self.world_model = WorldModel(obs_dim, config['world_dim'], hidden)
        
        # 9. Active inference
        self.active_inference = ActiveInference(latent_dim, n_met, hidden)
        
        # 10. Compositional metabolism
        self.compositional = CompositionalMetabolism(hidden)
        
        # Projection layers
        self.to_latent = nn.Linear(d_model, latent_dim)
        self.from_latent = nn.Linear(latent_dim, n_met + n_gene)
        
        # Output heads
        self.metabolite_head = nn.Linear(d_model, n_met)
        self.gene_head = nn.Linear(d_model, n_gene)
        self.uncertainty_head = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_met + n_gene),
            nn.Softplus()
        )
    
    def forward(self, metabolites, genes, perturbation, dt, use_ode=False):
        """
        Forward pass integrating all mechanisms.
        """
        batch_size = metabolites.shape[0]
        device = metabolites.device
        
        # 1. Transformer encoding
        encoded = self.transformer(metabolites, genes, perturbation)
        
        # Global state (mean pooling)
        global_state = encoded.mean(dim=1)  # (batch, d_model)
        
        # 4. Hyperbolic embedding (for hierarchy)
        hyp_embed = self.hyperbolic(global_state)
        
        # 5. Gene regulatory update
        new_genes, gene_effect = self.grn(genes, metabolites, global_state, dt)
        
        # 3. Enzyme kinetics
        velocities = self.enzyme_kinetics(metabolites, global_state)
        
        # 2. Neural ODE (optional - for long-horizon)
        if use_ode:
            z0 = self.to_latent(global_state)
            t_span = torch.tensor([0.0, dt], device=device)
            z_t = self.neural_ode(z0, t_span)
            z_final = z_t[-1]  # (batch, latent_dim)
            ode_contrib = self.from_latent(z_final)
        else:
            ode_contrib = torch.zeros(batch_size, self.config['n_metabolites'] + self.config['n_genes'], device=device)
        
        # 9. Active inference (goal-directed adjustment)
        z_current = self.to_latent(global_state)
        action, free_energy, expected_fe = self.active_inference(z_current)
        
        # Combine all contributions
        met_tokens = encoded[:, 1:1+self.config['n_metabolites'], :]  # Skip perturbation token
        gene_tokens = encoded[:, 1+self.config['n_metabolites']:, :]
        
        # Predictions
        met_pred = self.metabolite_head(met_tokens.mean(dim=1))
        met_pred = met_pred + gene_effect + 0.1 * velocities.sum(dim=-1, keepdim=True).expand_as(met_pred)
        met_pred = met_pred + action + ode_contrib[:, :self.config['n_metabolites']]
        
        gene_pred = self.gene_head(gene_tokens.mean(dim=1))
        gene_pred = 0.9 * new_genes + 0.1 * gene_pred
        
        # Clamp outputs
        new_metabolites = torch.clamp(metabolites + dt * (met_pred - metabolites), 0.05, 15.0)
        new_genes = torch.clamp(gene_pred, 0.01, 10.0)
        
        # Uncertainty
        uncertainty = self.uncertainty_head(global_state)
        
        return {
            'metabolites': new_metabolites,
            'genes': new_genes,
            'uncertainty': uncertainty,
            'free_energy': free_energy,
            'hyperbolic_embed': hyp_embed,
            'velocities': velocities,
            'encoded': encoded
        }
    
    def simulate(self, M0, G0, P_trajectory, times, dt=0.01):
        """
        Full trajectory simulation.
        """
        M = M0.clone()
        G = G0.clone()
        
        M_traj = [M0]
        G_traj = [G0]
        FE_traj = []
        
        t = 0.0
        for i, target in enumerate(times[1:], 1):
            P = P_trajectory[i]
            
            while t < target - 1e-6:
                step = min(dt, target - t)
                out = self.forward(M, G, P.unsqueeze(0).expand(M.shape[0], -1), step)
                M = out['metabolites']
                G = out['genes']
                t += step
            
            M_traj.append(M.clone())
            G_traj.append(G.clone())
            FE_traj.append(out['free_energy'].mean().item())
        
        return {
            'metabolites': torch.stack(M_traj, dim=1),
            'genes': torch.stack(G_traj, dim=1),
            'free_energy': FE_traj
        }

print("✅ DarkManifoldV21 (Biological Foundation Model) defined!")

In [ ]:
#@title 1️⃣4️⃣ Initialize V21

config = {
    'n_metabolites': N_METABOLITES,
    'n_genes': N_GENES,
    'perturbation_dim': PERTURBATION_DIM,
    'd_model': D_MODEL,
    'n_heads': N_HEADS,
    'n_transformer_layers': N_TRANSFORMER_LAYERS,
    'd_ff': D_FF,
    'dropout': DROPOUT,
    'latent_dim': LATENT_DIM,
    'hidden_dim': ODE_HIDDEN,
    'hyperbolic_dim': HYPERBOLIC_DIM,
    'world_dim': WORLD_DIM
}

model = DarkManifoldV21(config).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"\n🧬 V21 Biological Foundation Model initialized")
print(f"   Total parameters: {n_params:,}")
print(f"\n   Mechanisms:")
print(f"   ├── Transformer: {N_TRANSFORMER_LAYERS} layers × {N_HEADS} heads")
print(f"   ├── Neural ODE: {LATENT_DIM}D latent space")
print(f"   ├── Enzyme kinetics: Michaelis-Menten")
print(f"   ├── Hyperbolic: {HYPERBOLIC_DIM}D Poincaré ball")
print(f"   ├── Gene regulatory: {N_GENES} genes")
print(f"   ├── Causal discovery: do-calculus")
print(f"   ├── World model: {WORLD_DIM}D latent dreaming")
print(f"   ├── Active inference: free energy")
print(f"   └── Compositional: 5 pathway modules")

# 📌 V21 Summary

## What Makes V21 Special?

V21 is not just "V20 + more features". It represents a paradigm shift:

| Aspect | V16-V20 | V21 |
|--------|---------|-----|
| Learning | Pattern matching | Causal reasoning |
| Time | Discrete steps | Continuous ODE |
| Kinetics | Arbitrary functions | Michaelis-Menten |
| Hierarchy | Flat embedding | Hyperbolic geometry |
| Genes | Not modeled | Full GRN dynamics |
| Goals | None | Active inference |
| Adaptation | Retrain | Few-shot (MAML) |
| Modularity | Monolithic | Composable pathways |
| Predictions | Point estimates | Full world model |
| Interventions | Correlation | Causal do(X=x) |

## The Vision

V21 is designed to be a **foundation model** for cellular biology:

1. **Pre-train** on many organisms and conditions
2. **Fine-tune** with few shots for new organisms
3. **Compose** pathways from different sources
4. **Reason** about interventions (knockouts, drugs)
5. **Dream** future trajectories before committing

This is the architecture that could eventually become a true "cell GPT".